# ⚖️ Dynamic Legal Knowledge Retrieval for Judgment Prediction (DLKR-LJP)
### Inference-Time Retrieval + Dual-Metric Relevance Gating + `InLegalBERT` Prefix Fusion

**Abstract Implementation:**
- **Dynamic Retrieval Module (DRM):** Dynamically retrieves relevant statutory articles (IPC/BNS), essential legal ingredients, and landmark prior case precedents ahead of fusion.
- **Relevance and Confidence Gate (RCG):** Filters candidates using a dual-metric gate (semantic cosine threshold $\tau \ge 0.65$ + element consistency check) to eliminate noise.
- **Prefix Attention Fusion:** Projects gated legal knowledge into prefix tokens conditioning `law-ai/InLegalBERT` self-attention.
- **3-Head Multi-Task Prediction:**
  1. Law Articles Prediction (IPC 1860 & BNS 2023)
  2. Criminal Charges Classification
  3. Term of Penalty & Statutory Sentencing Severity Tiers

## 1. Verify GPU Hardware (T4 GPU / 15 GB VRAM)

In [1]:
!nvidia-smi

Wed Oct  7 17:24:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/DLKR_LJP_Indian_Law', exist_ok=True)
os.makedirs('/content/dlkr_ljp_runtime', exist_ok=True)
print("✅ Runtime directories initialized.")

Mounted at /content/drive
✅ Runtime directories initialized.


## 3. Install Dependencies

In [3]:
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 chromadb sentence-transformers gradio pydantic tabulate scikit-learn
print("✅ Dependencies successfully installed.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
✅ Dependencies successfully installed.


## 3.1 Unpack Project Codebase
Drag and drop `dynamic_legal_retrieval_ljp.zip` into the left Files tab in Colab or put it in Google Drive.

In [4]:
import os, zipfile

if not os.path.exists('/content/pipeline.py'):
    candidates = [
        '/content/dynamic_legal_retrieval_ljp.zip',
        '/content/drive/MyDrive/dynamic_legal_retrieval_ljp.zip',
        '/content/drive/MyDrive/DLKR_LJP_Indian_Law/dynamic_legal_retrieval_ljp.zip'
    ]
    found_zip = next((c for c in candidates if os.path.exists(c)), None)
    if found_zip:
        print(f'📦 Unpacking project from {found_zip}...')
        with zipfile.ZipFile(found_zip, 'r') as z:
            z.extractall('/content')
        print('✅ Codebase extracted successfully!')
    else:
        print('💡 Tip: Drag and drop dynamic_legal_retrieval_ljp.zip into Colab Files tab, then re-run this cell.')
else:
    print('✅ Project codebase already present in /content.')

📦 Unpacking project from /content/drive/MyDrive/dynamic_legal_retrieval_ljp.zip...
✅ Codebase extracted successfully!


## 4. Initialize DLKR-LJP Pipeline

In [5]:
import sys
if '.' not in sys.path:
    sys.path.append('.')

from pipeline import DynamicLjpPipeline

pipeline = DynamicLjpPipeline(load_transformer=True)
print("✅ DLKR-LJP Pipeline initialized with InLegalBERT backbone.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  534MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[DLKR-LJP] Successfully loaded law-ai/InLegalBERT.
✅ DLKR-LJP Pipeline initialized with InLegalBERT backbone.


## 5. Run Sample Case Adjudication (Cheating & Forgery)

In [6]:
sample_case = """
The accused persons dishonestly induced the complainant to transfer Rs. 85 Lakhs against promises of allotment of commercial land.
Investigation revealed that title deeds, town planning certificates, and official stamps produced by the accused were forged.
Cheques issued by the accused subsequently bounced due to closed accounts.
"""

result = pipeline.run(sample_case)

print("\n--- 1. RELEVANCE & CONFIDENCE GATE AUDIT ---")
print(f"Passed Gate Items (Fused into Prefix): {len(result['gated_knowledge'])}")
for item in result['gated_knowledge']:
    print(f"  [PASSED] {item['title']} (Similarity: {item['similarity']:.2f})")
print(f"Filtered Noise Items (Discarded): {len(result['filtered_noise'])}")
for item in result['filtered_noise']:
    print(f"  [REJECTED] {item['title']}: {item['reason']}")

print("\n--- 2. 3-HEAD PREDICTIONS ---")
print("Head 1 (Articles):", [(a['display_name'], f"{a['probability']:.1%}") for a in result['predictions']['head1_articles'][:3]])
print("Head 2 (Charges):", [(c['charge_name'], f"{c['probability']:.1%}") for c in result['predictions']['head2_charges'][:2]])
print("Head 3 (Penalty):", result['predictions']['primary_penalty']['tier'])

print("\n--- 3. TRACEABLE EVIDENCE ---")
print(result['traceable_evidence'])


[DLKR Pipeline] === Processing Legal Case Fact Narrative ===
[Step 1/4] Dynamically retrieving candidate articles, elements, and prior cases from external KB...
         Retrieved 6 candidate knowledge items.
[Step 2/4] Applying Relevance and Confidence Gate (Similarity Cutoff & Element Consistency)...
         Gate Decision: 3 PASSED | 3 DISCARDED AS NOISE.
[Step 3/4] Constructing gated prefix-attention token representations...
[Step 4/4] Executing 3-Head Multi-Task Prediction (Articles, Charges, Penalty)...


model.safetensors: reconstructing file:   0%|          |  0.00B /  534MB            

model.safetensors: downloading bytes:           |  0.00B            

[DLKR Pipeline] Adjudication complete.


--- 1. RELEVANCE & CONFIDENCE GATE AUDIT ---
Passed Gate Items (Fused into Prefix): 3
  [PASSED] Cheating and Dishonestly Inducing Delivery (IPC Section 420 / BNS Section 318(4)) (Similarity: 0.53)
  [PASSED] State of Haryana v. Bhajan Lal (1992) Supp (1) SCC 335 (Similarity: 0.49)
  [PASSED] Criminal Breach of Trust (IPC Section 406 / BNS Section 316) (Similarity: 0.43)
Filtered Noise Items (Discarded): 3
  [REJECTED] Forgery of Valuable Security or Will (IPC Section 467 / BNS Section 338): Low Confidence Gate: Similarity 0.383 is below effective threshold (0.40)
  [REJECTED] Using as Genuine a Forged Document (IPC Section 471 / BNS Section 340(2)): Capacity Gate: Exceeded max prefix budget slot limit
  [REJECTED] Forgery for Purpose of Cheating (IPC Section 468 / BNS Section 336(3)): Capacity Gate: Exceeded max prefix budget slot limit

--- 2. 3-HEAD PREDICTIONS ---
Head 1 (Articles): [('IPC 420 | BNS 318', '93.7%'), ('IPC 467 | BNS 338', '78.

## 6. Launch Interactive Gradio Dashboard (Public URL)

In [7]:
from app import demo
# share=True creates a public https://xxxx.gradio.live URL!
demo.launch(share=True)

/content/app.py:93: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="DLKR-LJP: Dynamic Legal Knowledge Retrieval", theme=theme) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://027d8a9d33ed88615b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
